<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/10-engineering-practices/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10: Engineering Practices

The first nine modules were about writing code that works. This one is about writing code that **keeps** working: when you come back to it in three months, when someone else runs it on their machine, when the data changes, and when you need it to run ten times faster.

None of this is specific to machine learning, but ML projects need it more than most. Results depend on library versions, random seeds and data files. Bugs rarely crash; they just make the model a bit worse. And experiments get run hundreds of times, so small inefficiencies add up.

**Contents**

1. From notebook to project
2. Environments and dependencies
3. A project layout
4. Testing with pytest
5. Logging instead of print
6. Errors and validation
7. Configuration and command-line scripts
8. Reproducibility
9. Profiling: finding what's slow
10. Habits that keep code healthy

Most cells in this lesson write small files into a `work/lesson10` folder and then run them, the way you would on your own machine. On Colab, the files appear in the file browser on the left.

In [1]:
import shutil
import sys
from pathlib import Path

import numpy as np

LESSON = Path("work/lesson10")
if LESSON.exists():
    shutil.rmtree(LESSON)                 # start clean every time the lesson is run
(LESSON / "ml_utils").mkdir(parents=True)
(LESSON / "tests").mkdir()
print("project folder:", LESSON)

project folder: work/lesson10


## 1. From notebook to project

Notebooks are excellent for exploring: you see results immediately, next to the code that made them. They're much worse as the final home for code you depend on:

- **Hidden state.** Cells can run in any order, and a variable can survive after the cell that made it was deleted. A notebook that works for you may fail when someone runs it top to bottom.
- **Hard to reuse.** To use a function from a notebook in another project, you copy and paste it, and now there are two versions to keep in sync.
- **Hard to test.** There's no standard way to run checks automatically after every change.
- **Hard to review.** Notebook files are JSON with embedded outputs, so changes are difficult to read in version control.

The usual pattern in real projects: **explore in notebooks, then move code that works into `.py` files**, and import it back into the notebook. The notebook becomes a thin layer that calls tested functions.

## 2. Environments and dependencies

Code doesn't run on its own. It runs on a particular Python version with particular library versions. Change those and results can change, or the code can stop working: you saw this in earlier modules, where a few behaviours differ between pandas 2 and pandas 3.

You can ask Python what it's running on:

In [2]:
from importlib.metadata import PackageNotFoundError, version

print("Python:", sys.version.split()[0])
print("inside a virtual environment:", sys.prefix != sys.base_prefix)
for package in ["numpy", "pandas", "matplotlib", "pytest"]:
    try:
        print(f"  {package:<12} {version(package)}")
    except PackageNotFoundError:
        print(f"  {package:<12} not installed")

Python: 3.13.18
inside a virtual environment: False
  numpy        2.5.3
  pandas       3.0.5
  matplotlib   3.11.2
  pytest       9.1.1


### Virtual environments

A **virtual environment** is a separate folder of installed packages for one project. Without one, every project on your machine shares the same packages, and upgrading NumPy for one project can break another.

On your own machine, the routine for a new project looks like this (in a terminal, not in Python):

```bash
python -m venv .venv                  # create the environment in a folder called .venv
source .venv/bin/activate             # activate it (Windows: .venv\Scripts\activate)
pip install -r requirements.txt       # install the project's dependencies into it
```

Once it's activated, `python` and `pip` refer to the environment's copies, and anything you install stays inside `.venv`. The `.venv` folder is never committed to git; the `requirements.txt` file is, so anyone can rebuild the same environment.

In a notebook, two details matter:

- Install with `%pip install package`, not `!pip install package`. `%pip` installs into the environment the notebook's kernel is actually using; `!pip` runs whichever `pip` is first on the system path, which may belong to a different Python.
- For the same reason, run Python tools from a notebook with `sys.executable`, the path of the running interpreter. You'll see `!{sys.executable} -m pytest` below.

Colab gives every session a fresh environment with common packages preinstalled, which is convenient, but its versions change over time. That's another reason to record versions (section 8).

### Pinning versions

There are two kinds of dependency list, with different jobs:

| file | example line | job |
|---|---|---|
| `requirements.txt` or `pyproject.toml` | `numpy>=1.26` | what the code **needs**: a range of versions that work |
| a lock file, e.g. from `pip freeze` | `numpy==2.0.2` | what you **actually used**: exact versions, for reproducing a result |

A library should state ranges, so it can be installed alongside other packages. An experiment you want to reproduce exactly should record exact versions. You'll write a parser for requirement lines in the exercises.

## 3. A project layout

A small ML project usually looks something like this:

```
penguin-project/
├── pyproject.toml        project name, dependencies and tool settings
├── README.md             what it is and how to run it
├── requirements.txt      (or dependencies listed in pyproject.toml)
├── penguin_project/      the importable package: your reusable code
│   ├── __init__.py
│   ├── data.py
│   └── model.py
├── tests/                one test file per module
│   ├── test_data.py
│   └── test_model.py
├── configs/              settings for experiments, as JSON or YAML
├── scripts/ or cli.py    command-line entry points: train, evaluate, predict
├── notebooks/            exploration, importing from the package
└── data/, models/        large files: in .gitignore, not in git
```

The important ideas are that **reusable code lives in a package**, **tests live next to it**, and **data, models and environments stay out of version control**.

Let's build a tiny version, a package called `ml_utils`. `%%writefile` at the top of a cell saves the cell's contents to a file instead of running it.

In [3]:
%%writefile work/lesson10/ml_utils/scaling.py
"""Feature scaling that remembers its training statistics."""
from __future__ import annotations

import numpy as np


class Standardizer:
    """Scale each column to mean 0 and standard deviation 1, using statistics from the training data."""

    def __init__(self) -> None:
        self.mean_: np.ndarray | None = None
        self.std_: np.ndarray | None = None

    def fit(self, X) -> "Standardizer":
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError(f"expected a 2D array (rows x columns), got shape {X.shape}")
        self.mean_ = X.mean(axis=0)
        std = X.std(axis=0)
        self.std_ = np.where(std == 0, 1.0, std)      # constant column: centre it, don't divide by 0
        return self

    def transform(self, X) -> np.ndarray:
        if self.mean_ is None:
            raise RuntimeError("call fit() before transform()")
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.mean_.shape[0]:
            raise ValueError(f"expected {self.mean_.shape[0]} columns, got shape {X.shape}")
        return (X - self.mean_) / self.std_

    def fit_transform(self, X) -> np.ndarray:
        return self.fit(X).transform(X)

Writing work/lesson10/ml_utils/scaling.py


Here's a first version of a metrics module. It has a bug that's easy to miss; we'll let a test find it in section 4.

In [4]:
%%writefile work/lesson10/ml_utils/metrics.py
"""Evaluation metrics."""


def accuracy(y_true, y_pred) -> float:
    """Fraction of predictions that match the true labels."""
    correct = sum(t == p for t, p in zip(y_true, y_pred))
    return correct / len(y_true)

Writing work/lesson10/ml_utils/metrics.py


In [5]:
%%writefile work/lesson10/ml_utils/__init__.py
"""Small utilities for ML experiments."""
from .metrics import accuracy
from .scaling import Standardizer

__version__ = "0.1.0"

Writing work/lesson10/ml_utils/__init__.py


`__init__.py` marks the folder as a package and decides what `import ml_utils` exposes. Importing the two names here lets users write `from ml_utils import Standardizer` instead of `from ml_utils.scaling import Standardizer`.

A `pyproject.toml` describes the project. This one is minimal; the `[tool.pytest.ini_options]` section tells pytest where to find the package when it runs the tests.

In [6]:
%%writefile work/lesson10/pyproject.toml
[project]
name = "ml-utils"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["numpy>=1.26"]

[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]

Writing work/lesson10/pyproject.toml


To use the package from this notebook, the folder that **contains** it has to be on Python's import path. In a real project you'd run `pip install -e .` once in the project folder, which installs it in "editable" mode: Python imports straight from your source files, so edits take effect without reinstalling. Here, adding the folder to `sys.path` does the same job for this session:

In [7]:
sys.path.insert(0, str(LESSON))

%load_ext autoreload
%autoreload 2

from ml_utils import Standardizer, accuracy

X = np.array([[1.0, 200.0], [2.0, 220.0], [3.0, 240.0]])
print(Standardizer().fit_transform(X).round(3))
print("accuracy:", accuracy([1, 0, 1, 1], [1, 0, 0, 1]))

[[-1.225 -1.225]
 [ 0.     0.   ]
 [ 1.225  1.225]]
accuracy: 0.75


`%autoreload 2` is worth knowing about. Normally Python imports a module once and ignores later changes to the file. With autoreload on, the notebook picks up your edits to `.py` files before running each cell, so you can edit the package and the notebook side by side.

## 4. Testing with pytest

A **test** is a function that runs your code and checks the result with `assert`. **pytest** finds every function whose name starts with `test_` in files named `test_*.py`, runs them all, and reports which ones failed and why. It's the standard testing tool in Python and is preinstalled on Colab.

Good tests check **behaviour you care about**, one idea per test, with names that say what should be true:

In [8]:
%%writefile work/lesson10/tests/test_scaling.py
import numpy as np
import pytest

from ml_utils import Standardizer


@pytest.fixture
def data():
    """Three columns on very different scales."""
    rng = np.random.default_rng(0)
    return rng.normal(loc=[10, -5, 0], scale=[2, 0.5, 100], size=(200, 3))


def test_training_columns_have_mean_0_and_std_1(data):
    Z = Standardizer().fit_transform(data)
    assert Z.mean(axis=0) == pytest.approx(0, abs=1e-12)
    assert Z.std(axis=0) == pytest.approx(1)


def test_new_data_uses_training_statistics(data):
    scaler = Standardizer().fit(data)
    new = data[:5] + 1.0
    expected = (new - data.mean(axis=0)) / data.std(axis=0)
    assert scaler.transform(new) == pytest.approx(expected)


def test_constant_column_gives_zeros_not_nan():
    X = np.array([[1.0, 7.0], [2.0, 7.0], [3.0, 7.0]])
    Z = Standardizer().fit_transform(X)
    assert np.all(np.isfinite(Z))
    assert Z[:, 1] == pytest.approx(0)


def test_input_is_not_modified(data):
    before = data.copy()
    Standardizer().fit_transform(data)
    assert np.array_equal(data, before)


def test_transform_before_fit_raises():
    with pytest.raises(RuntimeError, match="fit"):
        Standardizer().transform(np.ones((2, 2)))


@pytest.mark.parametrize("bad", [np.ones(3), np.ones((2, 2, 2))])
def test_rejects_input_that_is_not_2d(bad):
    with pytest.raises(ValueError):
        Standardizer().fit(bad)


def test_rejects_wrong_number_of_columns(data):
    scaler = Standardizer().fit(data)
    with pytest.raises(ValueError, match="columns"):
        scaler.transform(np.ones((4, 2)))

Writing work/lesson10/tests/test_scaling.py


The tools used there:

- **`pytest.approx`** compares floats with a tolerance. Never test floats with `==` (Module 9).
- **`pytest.raises`** checks that an error **is** raised. `match=` checks the message too, so a test can't pass because of some unrelated error.
- **A fixture** (`@pytest.fixture`) builds test data once and passes it to any test that names it as an argument. A seeded generator makes it the same every run.
- **`@pytest.mark.parametrize`** runs one test with several inputs, each reported separately.

And the metrics tests. One of them checks what happens with mismatched lengths, a mistake that's easy to make when filtering predictions and labels separately:

In [9]:
%%writefile work/lesson10/tests/test_metrics.py
import pytest

from ml_utils import accuracy


def test_perfect_predictions():
    assert accuracy([0, 1, 2], [0, 1, 2]) == 1.0


def test_known_value():
    assert accuracy([1, 0, 1, 1], [1, 0, 0, 1]) == pytest.approx(0.75)


def test_mismatched_lengths_raise():
    with pytest.raises(ValueError):
        accuracy([1, 0, 1], [1, 0])


def test_empty_input_raises():
    with pytest.raises(ValueError):
        accuracy([], [])

Writing work/lesson10/tests/test_metrics.py


Run the whole suite. `cd` into the project folder first, as you would in a terminal, so pytest finds its settings in `pyproject.toml`. `-q` keeps the output short.

In [10]:
!cd {LESSON} && {sys.executable} -m pytest -q

..F

F........                                                             [100%]
=================================== FAILURES ===================================
________________________ test_mismatched_lengths_raise _________________________

    def test_mismatched_lengths_raise():
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E       Failed: DID NOT RAISE ValueError

tests/test_metrics.py:15: Failed
___________________________ test_empty_input_raises ____________________________

    def test_empty_input_raises():
        with pytest.raises(ValueError):
>           accuracy([], [])

tests/test_metrics.py:21: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

y_true = [], y_pred = []

    def accuracy(y_true, y_pred) -> float:
        """Fraction of predictions that match the true labels."""
        correct = sum(t == p for t, p in zip(y_true, y_pred))
>       return correct / len(y_true)
               ^^^^^^^^^^^^^^^^^^^^

The scaling tests pass, and two metrics tests fail. The report shows exactly what went wrong:

- **Mismatched lengths:** no error was raised. `zip` stops at the shorter list, so `accuracy([1, 0, 1], [1, 0])` quietly compared two pairs and divided by 3, returning 0.67 without complaint.
- **Empty input:** the error raised was `ZeroDivisionError`, not the `ValueError` the test asked for. A crash deep inside a function is much harder to understand than a clear message at the start.

Fix the function: check the inputs first, then compute.

In [11]:
%%writefile work/lesson10/ml_utils/metrics.py
"""Evaluation metrics."""
import numpy as np


def accuracy(y_true, y_pred) -> float:
    """Fraction of predictions that match the true labels."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"y_true has shape {y_true.shape} but y_pred has shape {y_pred.shape}")
    if y_true.size == 0:
        raise ValueError("cannot compute accuracy of empty arrays")
    return float(np.mean(y_true == y_pred))

Overwriting work/lesson10/ml_utils/metrics.py


In [12]:
!cd {LESSON} && {sys.executable} -m pytest -q

............                                                             [100%]
12 passed in 0.07s


All green. That's the basic loop of test-driven work: **when you find a bug, first write a test that fails because of it, then fix the code**. The test stays in the suite forever, so the bug can't come back unnoticed.

What's worth testing in ML code:

- **Shapes and types** of outputs.
- **Known values** worked out by hand on tiny inputs.
- **Invariants**: probabilities sum to 1, a loss is never negative, standardised training columns have mean 0.
- **Edge cases**: empty input, a single row, a constant column, missing values.
- **No side effects**: inputs aren't modified.
- **Errors**: bad input raises a clear error instead of returning nonsense.
- **Reproducibility**: the same seed gives the same result.

You can't easily test that a model is "good", but you can test that every piece behaves correctly, which removes most of the ways it can be quietly bad.

## 5. Logging instead of print

`print` is fine for exploring. For code that runs unattended, such as a training job, use the `logging` module. It adds a timestamp and a **level** to every message, can send messages to the screen and a file at the same time, and lets you turn detail up or down without editing the code.

The five levels, from least to most serious: `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`. A logger only shows messages at or above its level.

In [13]:
import logging

logger = logging.getLogger("lesson10")
logger.setLevel(logging.INFO)
logger.propagate = False                       # don't also pass messages to the root logger

if not logger.handlers:                        # re-running this cell mustn't add a second handler
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)-7s %(name)s: %(message)s", datefmt="%H:%M:%S"))
    logger.addHandler(handler)

logger.debug("hidden: DEBUG is below the INFO level")
logger.info("loaded %d rows from %s", 333, "penguins.csv")
logger.warning("%d rows had missing values and were dropped", 11)

05:23:36 INFO    lesson10: loaded 333 rows from penguins.csv


05:23:36 WARNING lesson10: 11 rows had missing values and were dropped


Three habits:

- **Pass values as arguments**, `logger.info("loaded %d rows", n)`, not with an f-string. The text is only built if the message is actually shown, and log tools can group messages that share a template.
- **Guard against duplicate handlers.** In a notebook, running the setup cell twice would otherwise print every message twice. It's the most common logging surprise.
- **Libraries don't configure logging.** Code in a package just does `logger = logging.getLogger(__name__)` and logs. The application that uses it (a script, a notebook) decides the format, the level and where messages go.

Adding a file handler keeps a permanent record of a run, which is what you want for a training job that runs overnight:

In [14]:
log_file = LESSON / "train.log"
file_handler = logging.FileHandler(log_file, mode="w")
file_handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
file_handler.setLevel(logging.DEBUG)
logger.addHandler(file_handler)
logger.setLevel(logging.DEBUG)                  # the logger passes everything; each handler filters

for epoch in range(1, 4):
    logger.debug("epoch %d: batch losses look normal", epoch)
    logger.info("epoch %d: loss %.4f", epoch, 1 / epoch)

try:
    threshold = float("0,5")                    # a decimal comma from a config written in another locale
except ValueError:
    logger.exception("could not read the threshold")   # logs at ERROR level, with the full traceback

logger.removeHandler(file_handler)
file_handler.close()
logger.handlers[0].setLevel(logging.INFO)

print("\n--- contents of train.log ---")
print(log_file.read_text())

05:23:36 DEBUG   lesson10: epoch 1: batch losses look normal


05:23:36 INFO    lesson10: epoch 1: loss 1.0000


05:23:36 DEBUG   lesson10: epoch 2: batch losses look normal


05:23:36 INFO    lesson10: epoch 2: loss 0.5000


05:23:36 DEBUG   lesson10: epoch 3: batch losses look normal


05:23:36 INFO    lesson10: epoch 3: loss 0.3333


05:23:36 ERROR   lesson10: could not read the threshold
Traceback (most recent call last):
  File "/tmp/ipykernel_1248/2687374630.py", line 13, in <module>
    threshold = float("0,5")                    # a decimal comma from a config written in another locale
ValueError: could not convert string to float: '0,5'



--- contents of train.log ---
2026-11-05 05:23:36,990 DEBUG epoch 1: batch losses look normal
2026-11-05 05:23:36,991 INFO epoch 1: loss 1.0000
2026-11-05 05:23:36,992 DEBUG epoch 2: batch losses look normal
2026-11-05 05:23:36,992 INFO epoch 2: loss 0.5000
2026-11-05 05:23:36,993 DEBUG epoch 3: batch losses look normal
2026-11-05 05:23:36,994 INFO epoch 3: loss 0.3333
2026-11-05 05:23:36,995 ERROR could not read the threshold
Traceback (most recent call last):
  File "/tmp/ipykernel_1248/2687374630.py", line 13, in <module>
    threshold = float("0,5")                    # a decimal comma from a config written in another locale
ValueError: could not convert string to float: '0,5'



The screen got the `INFO` messages; the file got everything, including `DEBUG` lines and the traceback. Each **handler** has its own level, so you can keep the screen readable and the file detailed.

## 6. Errors and validation

Bad data is the normal state of affairs in ML. The question is where you find out. Compare:

- **Fail late:** a column of strings reaches the model, NumPy makes an `object` array, and twenty lines later something raises `TypeError: unsupported operand type(s) for -: 'str' and 'float'`.
- **Fail early:** the first function that receives the data checks it and says `column 'body_mass_g' should be numeric, but contains '4,050'`.

The second takes seconds to fix. Validate at the **boundaries** of your code, where data comes in from files, users or other systems, and raise errors that name the problem and the value.

In [15]:
class DataValidationError(ValueError):
    """Raised when input data doesn't meet a function's requirements."""


def check_features(X, n_features: int) -> np.ndarray:
    try:
        X = np.asarray(X, dtype=float)
    except ValueError as e:
        raise DataValidationError(f"features must be numeric: {e}") from e
    if X.ndim != 2 or X.shape[1] != n_features:
        raise DataValidationError(f"expected shape (n, {n_features}), got {X.shape}")
    if not np.all(np.isfinite(X)):
        bad_rows = np.unique(np.argwhere(~np.isfinite(X))[:, 0])
        raise DataValidationError(f"{len(bad_rows)} row(s) contain NaN or infinity, first at row {bad_rows[0]}")
    return X


for bad in [[["39.1", "4,050"]], np.ones((3, 2)), [[39.1, np.nan, 181.0], [40.0, 18.0, 190.0]]]:
    try:
        check_features(bad, n_features=3)
    except DataValidationError as e:
        print("DataValidationError:", e)

DataValidationError: features must be numeric: could not convert string to float: '4,050'
DataValidationError: expected shape (n, 3), got (3, 2)
DataValidationError: 1 row(s) contain NaN or infinity, first at row 0


Some guidelines:

- **A custom exception class** that subclasses a built-in one (`ValueError` here) lets callers catch your errors specifically, while code that catches `ValueError` still works.
- **`raise ... from e`** keeps the original error attached, so the traceback shows both.
- **Never use a bare `except:`** (or `except Exception:` that does nothing). It hides every bug, including typos in your own code. Catch the specific errors you expect and can handle.
- **`assert` is for your own assumptions, not for input checks.** Python skips `assert` statements entirely when run with `python -O`, so they can't be relied on to reject bad data. Use `if ...: raise`.

## 7. Configuration and command-line scripts

Hard-coded settings scattered through code are hard to find and change, and impossible to record. Collect them in one **config** object, which can be saved next to the results so every run documents itself.

A frozen dataclass makes a good config: typed fields, defaults, no accidental changes, and a `__post_init__` hook for validation.

In [16]:
import json
from dataclasses import asdict, dataclass, fields


@dataclass(frozen=True)
class TrainConfig:
    lr: float = 0.1
    epochs: int = 200
    seed: int = 0
    test_fraction: float = 0.25

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f"lr must be positive, got {self.lr}")
        if self.epochs < 1:
            raise ValueError(f"epochs must be at least 1, got {self.epochs}")
        if not 0 < self.test_fraction < 1:
            raise ValueError(f"test_fraction must be between 0 and 1, got {self.test_fraction}")

    @classmethod
    def from_json(cls, path) -> "TrainConfig":
        values = json.loads(Path(path).read_text())
        unknown = set(values) - {f.name for f in fields(cls)}
        if unknown:
            raise KeyError(f"unknown config keys: {sorted(unknown)}")
        return cls(**values)


config_path = LESSON / "config.json"
config_path.write_text(json.dumps({"lr": 0.5, "epochs": 300}, indent=2))
config = TrainConfig.from_json(config_path)
print(config)
print(asdict(config))

TrainConfig(lr=0.5, epochs=300, seed=0, test_fraction=0.25)
{'lr': 0.5, 'epochs': 300, 'seed': 0, 'test_fraction': 0.25}


The unknown-keys check is the most valuable line. Without it, a typo in a config file is silently ignored, and you run an experiment with the default setting while believing you changed it:

In [17]:
config_path.write_text(json.dumps({"learning_rate": 0.5}))
try:
    TrainConfig.from_json(config_path)
except KeyError as e:
    print("KeyError:", e)

KeyError: "unknown config keys: ['learning_rate']"


### A command-line script

Training runs usually happen from a terminal, a scheduler or a cloud job, not a notebook. `argparse` (in the standard library) turns a script's settings into command-line options with automatic `--help` text. Here's a complete small training script: it reads a config, lets the command line override it, logs its progress and saves its results.

In [18]:
%%writefile work/lesson10/train.py
"""Fit a linear model with gradient descent on synthetic data, and save the results."""
import argparse
import json
import logging
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np

from ml_utils import Standardizer

log = logging.getLogger("train")


@dataclass(frozen=True)
class Config:
    lr: float = 0.1
    epochs: int = 200
    seed: int = 0


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, help="JSON file with settings")
    parser.add_argument("--lr", type=float, help="learning rate (overrides the config file)")
    parser.add_argument("--epochs", type=int, help="number of epochs (overrides the config file)")
    parser.add_argument("--out", type=Path, default=Path("runs/latest"), help="folder for results")
    return parser.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

    config = Config(**json.loads(args.config.read_text())) if args.config else Config()
    overrides = {k: v for k, v in {"lr": args.lr, "epochs": args.epochs}.items() if v is not None}
    config = replace(config, **overrides)
    log.info("config: %s", asdict(config))

    rng = np.random.default_rng(config.seed)
    X = rng.normal(loc=[50, 2], scale=[10, 0.5], size=(500, 2))
    y = X @ np.array([0.3, -4.0]) + 7 + rng.normal(scale=0.5, size=500)

    Z = Standardizer().fit_transform(X)
    w, b = np.zeros(2), 0.0
    for epoch in range(config.epochs):
        error = Z @ w + b - y
        w -= config.lr * 2 * Z.T @ error / len(y)
        b -= config.lr * 2 * error.mean()
        if epoch % 100 == 0:
            log.info("epoch %d: mse %.4f", epoch, np.mean(error ** 2))

    mse = float(np.mean((Z @ w + b - y) ** 2))
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "results.json").write_text(json.dumps({"config": asdict(config), "mse": mse}, indent=2))
    log.info("final mse %.4f, saved to %s", mse, args.out / "results.json")


if __name__ == "__main__":
    main()

Writing work/lesson10/train.py


Two details make this script easy to work with:

- **`if __name__ == "__main__":`** runs `main()` only when the file is executed as a script, not when it's imported. Tests can import the file and call `main([...])` with a list of arguments.
- **`main(argv=None)`** passes `argv` to `parse_args`. `None` means "read the real command line"; a list lets tests and notebooks pass arguments directly.

The `--help` text is generated from the arguments:

In [19]:
!{sys.executable} {LESSON}/train.py --help

usage: train.py [-h] [--config CONFIG] [--lr LR] [--epochs EPOCHS] [--out OUT]

Fit a linear model with gradient descent on synthetic data, and save the
results.

options:
  -h, --help       show this help message and exit
  --config CONFIG  JSON file with settings
  --lr LR          learning rate (overrides the config file)
  --epochs EPOCHS  number of epochs (overrides the config file)
  --out OUT        folder for results


Run it with the config file from before, overriding the number of epochs on the command line:

In [20]:
config_path.write_text(json.dumps({"lr": 0.05}))
!cd {LESSON} && {sys.executable} train.py --config config.json --epochs 300 --out runs/lr0.05
print((LESSON / "runs/lr0.05/results.json").read_text())

INFO train: config: {'lr': 0.05, 'epochs': 300, 'seed': 0}
INFO train: epoch 0: mse 202.6791
INFO train: epoch 100: mse 0.2543
INFO train: epoch 200: mse 0.2543
INFO train: final mse 0.2543, saved to runs/lr0.05/results.json


{
  "config": {
    "lr": 0.05,
    "epochs": 300,
    "seed": 0
  },
  "mse": 0.2542909935211838
}


The results file records the exact settings that produced it, including the override. Six months later, that's the difference between "I think I used 300 epochs" and knowing.

## 8. Reproducibility

A result you can't reproduce is hard to trust and impossible to debug. Three things change results between runs: **randomness**, **software versions** and **data**.

### Randomness

NumPy has two ways to generate random numbers. The old one, `np.random.seed()` with `np.random.rand()` and friends, uses one **global** generator shared by all code. That makes results depend on everything else that ran before:

In [21]:
def noisy_split(n):
    """Pick 3 test rows using the global random state."""
    return np.random.permutation(n)[:3]


np.random.seed(0)
print("run 1:", noisy_split(10))

np.random.seed(0)
_ = np.random.rand(3)                # some unrelated code also draws random numbers...
print("run 2:", noisy_split(10))     # ...and the "seeded" split has changed

run 1: [2 8 4]
run 2: [8 6 4]


Same seed, different split, because another piece of code consumed random numbers in between. In a large program, that other code can be anywhere, even inside a library.

The modern approach, used throughout this course, is to create a **`Generator`** and pass it explicitly to whatever needs randomness. Nothing else can disturb it:

In [22]:
def clean_split(n, rng):
    return rng.permutation(n)[:3]


print("run 1:", clean_split(10, np.random.default_rng(0)))
_ = np.random.rand(3)
print("run 2:", clean_split(10, np.random.default_rng(0)))

run 1: [4 6 2]
run 2: [4 6 2]


Libraries follow the same idea: scikit-learn functions take a `random_state` argument, and PyTorch has its own generator (`torch.manual_seed`). Some sources of variation can't be seeded away, such as the order of parallel operations on a GPU, so deep learning results can differ in the last few digits even with every seed set.

### Versions and data

Record what produced a result, next to the result: the Python and package versions, and a **fingerprint** of the data file. A SHA-256 hash changes completely if even one byte of the file changes, so it shows whether two runs really used the same data.

In [23]:
import hashlib
import platform


def file_fingerprint(path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]


data_file = LESSON / "features.csv"
np.savetxt(data_file, np.random.default_rng(0).normal(size=(100, 3)), delimiter=",", fmt="%.6f")

run_record = {
    "config": asdict(TrainConfig(lr=0.5)),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "data_file": data_file.name,
    "data_sha256": file_fingerprint(data_file),
}
print(json.dumps(run_record, indent=2))

{
  "config": {
    "lr": 0.5,
    "epochs": 200,
    "seed": 0,
    "test_fraction": 0.25
  },
  "python": "3.13.18",
  "numpy": "2.5.3",
  "data_file": "features.csv",
  "data_sha256": "089e47aa55e731f1"
}


In the exercises you'll write a version that reads large files in chunks, so it doesn't need to load a multi-gigabyte file into memory at once.

The general rule: **save the config, the versions and the data fingerprint with every result.** It costs a few lines and makes every result traceable.

## 9. Profiling: finding what's slow

When code is too slow, the tempting move is to guess which part is slow and rewrite it. Guesses are usually wrong. **Measure first.**

Here's a deliberately naive nearest-neighbour classifier, written with Python loops:

In [24]:
def distance(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5


def knn_predict_slow(X_train, y_train, X_test, k=5):
    predictions = []
    for x in X_test:
        dists = [distance(x, t) for t in X_train]
        nearest = np.argsort(dists)[:k]
        predictions.append(np.bincount(y_train[nearest]).argmax())
    return np.array(predictions)


rng = np.random.default_rng(1)
X_train = rng.normal(size=(1000, 4))
y_train = (X_train[:, 0] + X_train[:, 1] > 0).astype(int)
X_test = rng.normal(size=(200, 4))

**`cProfile`**, in the standard library, records every function call and how long it took. `pstats` sorts and prints the results:

In [25]:
import cProfile
import pstats

profiler = cProfile.Profile()
slow_predictions = profiler.runcall(knn_predict_slow, X_train, y_train, X_test)

stats = pstats.Stats(profiler).strip_dirs().sort_stats("tottime")
stats.print_stats(5);

         1402535 function calls (1402534 primitive calls) in 0.811 seconds

   Ordered by: internal time
   List reduced from 38 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  1000000    0.379    0.000    0.379    0.000 71220894.py:2(<genexpr>)
   200000    0.229    0.000    0.607    0.000 {built-in method builtins.sum}
   200000    0.127    0.000    0.734    0.000 71220894.py:1(distance)
      2/1    0.064    0.032    0.782    0.782 71220894.py:5(knn_predict_slow)
      200    0.006    0.000    0.011    0.000 fromnumeric.py:38(_wrapit)




How to read it:

- **`ncalls`**: how many times the function was called. `distance` was called 200,000 times (200 test points × 1,000 training points), and the generator expression inside it a million times.

You may also see an entry or two from Jupyter itself, such as `selectors.py`. Those are the notebook waiting for input in the background, not part of your code. The semicolon at the end of the cell stops the notebook from also printing the `Stats` object that `print_stats` returns.
- **`tottime`**: time spent **inside** that function itself, not counting functions it called.
- **`cumtime`**: total time including everything it called.

The time goes into `distance`, the generator inside it and `sum`: Python-level arithmetic, one number at a time. The sorting (`argsort`) and voting (`bincount`) barely register. So that's what to fix, with the NumPy broadcasting from Module 6:

In [26]:
def knn_predict_fast(X_train, y_train, X_test, k=5):
    diffs = X_test[:, None, :] - X_train[None, :, :]           # shape (n_test, n_train, d)
    dists = np.sqrt((diffs ** 2).sum(axis=2))                   # shape (n_test, n_train)
    nearest = np.argsort(dists, axis=1)[:, :k]
    votes = y_train[nearest]
    return (votes.mean(axis=1) > 0.5).astype(int)               # majority of k=5 binary votes


fast_predictions = knn_predict_fast(X_train, y_train, X_test)
print("same predictions:", np.array_equal(slow_predictions, fast_predictions))

same predictions: True


**Check that the fast version gives the same answer before timing it.** A fast wrong function is worse than a slow right one. In a real project, this comparison would be a test.

Now time both. `time.perf_counter()` is the right clock for measuring durations; repeating and taking the best run reduces noise from whatever else the machine is doing:

In [27]:
import time


def best_time(func, *args, repeats=3):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        func(*args)
        times.append(time.perf_counter() - start)
    return min(times)


t_slow = best_time(knn_predict_slow, X_train, y_train, X_test)
t_fast = best_time(knn_predict_fast, X_train, y_train, X_test)
print(f"slow: {t_slow:.3f} s   fast: {t_fast:.4f} s   speed-up: about {t_slow / t_fast:.0f}x")

slow: 0.421 s   fast: 0.0069 s   speed-up: about 61x


In a notebook, `%timeit` does the repeating for you and picks a sensible number of runs:

In [28]:
%timeit knn_predict_fast(X_train, y_train, X_test)

6.41 ms ± 179 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Speed has a price: memory

The fast version builds a `(200, 1000, 4)` array of differences. That's fine here, but the size grows with `n_test × n_train`. `tracemalloc` measures the peak memory a piece of code allocates:

In [29]:
import tracemalloc


def peak_memory_mb(func, *args):
    tracemalloc.start()
    func(*args)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1e6


print(f"200 test points:   {peak_memory_mb(knn_predict_fast, X_train, y_train, X_test):6.1f} MB")
big_test = rng.normal(size=(2000, 4))
print(f"2,000 test points: {peak_memory_mb(knn_predict_fast, X_train, y_train, big_test):6.1f} MB")

200 test points:     14.4 MB
2,000 test points:  144.0 MB


Ten times the test points, ten times the memory. With 100,000 test points it would need several gigabytes. The standard fix is to process the test set in **batches** of, say, 1,000 rows: almost all of the speed, with memory that stays bounded. That trade-off between speed and memory comes up constantly, and it's the same reason neural networks are trained in mini-batches.

### Don't compute the same thing twice

When a slow function is called repeatedly with the same arguments, `functools.lru_cache` remembers the results:

In [30]:
from functools import lru_cache


@lru_cache(maxsize=None)
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)


start = time.perf_counter()
print(fib(200))
print(f"took {time.perf_counter() - start:.5f} s;", fib.cache_info())

280571172992510140037611932413038677189525
took 0.00044 s; CacheInfo(hits=198, misses=201, maxsize=None, currsize=201)


Without the cache, `fib(200)` would make around 10<sup>42</sup> calls and take far longer than the age of the universe. With it, each value is computed once. Caching only works for functions whose arguments are hashable (numbers, strings, tuples) and whose results depend only on their arguments. You'll build a small cache class of your own in the exercises.

**The order of work:** make it correct (with tests), then measure, then speed up the part that's actually slow, then check it's still correct.

## 10. Habits that keep code healthy

A few practices that cost little and pay off in every project:

**Type hints and docstrings.** `def accuracy(y_true, y_pred) -> float:` with a one-line docstring tells readers (and editors like VS Code) what goes in and what comes out. They don't change how the code runs, but they make mistakes visible before running it.

**A formatter and a linter.** Tools like **ruff** (`ruff format` and `ruff check`) or **black** format code consistently and catch common mistakes such as unused imports and undefined names. Run them before committing and you never argue about formatting again.

**Version control with git**, committing small, working changes with messages that say why. And a **`.gitignore`** so that things that shouldn't be in the repository never get there:

```
.venv/              # environments: rebuilt from requirements.txt
__pycache__/        # compiled Python files
.ipynb_checkpoints/ # Jupyter autosaves
data/raw/           # large or private data
models/  *.pt       # trained weights
work/               # scratch output
.env                # secrets: API keys, passwords
```

Never commit passwords or API keys, even to a private repository. Read them from environment variables (`os.environ["API_KEY"]`) or a `.env` file that's in `.gitignore`. Once a secret is pushed, assume it's public and replace it.

**Continuous integration.** Services like GitHub Actions can run your test suite automatically on every push, so a broken change is caught within minutes, before anyone else pulls it.

## Summary

- **Explore in notebooks; move working code into a package** of `.py` files and import it back. Use `%autoreload 2` while editing.
- **One virtual environment per project.** List dependency ranges in `requirements.txt` or `pyproject.toml`; record exact versions for reproducible results. In notebooks use `%pip` and `sys.executable`.
- **Layout:** a package folder, a `tests/` folder, configs, and data, models and environments kept out of git.
- **pytest:** functions named `test_*` with plain `assert`, `pytest.approx` for floats, `pytest.raises` for errors, fixtures for shared data, `parametrize` for several inputs. When you find a bug, write the failing test first.
- **logging:** levels, a handler per destination, values as arguments, no duplicate handlers. Libraries log; applications configure.
- **Validate at the boundaries** with clear errors that name the problem. Specific exceptions, never a bare `except`, no `assert` for input checks.
- **Config in one place** (a frozen dataclass), checked for unknown keys and saved with the results. `argparse` with `main(argv=None)` for scripts.
- **Reproducibility:** pass a seeded `Generator` instead of using global random state; save versions and a data fingerprint with every result.
- **Profile before optimising** (`cProfile`, `perf_counter`, `%timeit`), check the fast version gives the same answer, and watch memory (`tracemalloc`). Cache repeated work.

Next, open `02_exercises.ipynb`.